## Classes

In [19]:
# Network
from torch import nn
from typing import Literal



class MLPClassifier(nn.Module):
    def __init__(self, n_hidden:int, hidden_dim:int, input_dim:int, output_dim:int, activation:Literal['ReLU', 'Tanh'] = 'ReLU'):
        super(MLPClassifier, self).__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

        if activation == 'ReLU':
            self.activation = nn.ReLU()
        elif activation == 'Tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError(f"Unknown activation function: {activation}")

    def forward(self, input):
        x = self.activation(self.input_layer(input))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.output_layer(x)

        return x

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from typing import Union, Optional, List, Dict, Any

DataType = Union[pd.DataFrame, pd.Series, np.ndarray]

class MyDataset(Dataset):
    def __init__(self, features: DataType, target: DataType) -> None:
        if isinstance(features, (pd.DataFrame, pd.Series)):
            features = features.values
        if isinstance(target, (pd.DataFrame, pd.Series)):
            target = target.values

        self.data: torch.Tensor = torch.tensor(features, dtype=torch.float32)
        self.outputs: torch.Tensor = torch.tensor(target, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.data[idx], self.outputs[idx]


class MyDataModule(pl.LightningDataModule):
    def __init__(self, X: DataType, y: DataType, batch_size: int = 32) -> None:
        super().__init__()
        self.X: DataType = X
        self.y: DataType = y
        self.batch_size: int = batch_size

        self.train_dataset: Optional[MyDataset] = None
        self.val_dataset: Optional[MyDataset] = None
        self.test_dataset: Optional[MyDataset] = None
        self.sampler: Optional[WeightedRandomSampler] = None

    def setup(self, stage: Optional[str] = None) -> None:
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            self.X,
            self.y,
            test_size=0.2,
            random_state=42
        )

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full,
            y_train_full,
            test_size=0.2,
            random_state=42
        )

        self.train_dataset = MyDataset(X_tr, y_tr)
        self.val_dataset = MyDataset(X_val, y_val)
        self.test_dataset = MyDataset(X_test, y_test)

        y_tr_series: pd.Series = pd.Series(y_tr) if isinstance(y_tr, np.ndarray) else y_tr

        class_counts: Dict[Any, int] = y_tr_series.value_counts().to_dict()
        class_weights: Dict[Any, float] = {cls: 1.0 / count for cls, count in class_counts.items()}
        sample_weights: List[float] = [class_weights[sample] for sample in y_tr_series]

        self.sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(self.train_dataset, batch_size=self.batch_size, sampler=self.sampler)

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self) -> DataLoader:
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)

In [21]:
from torch import nn

# Lit Model
class LitClassificationModel(pl.LightningModule):
    def __init__(self, n_hidden:int, hidden_dim:int, input_dim:int, output_dim:int, lr:float = 1e-3):
        super(LitClassificationModel, self).__init__()
        self.save_hyperparameters()

        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = nn.ReLU()

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input):
        x = self.activation(self.input_layer(input))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.output_layer(x)

        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)

        preds = torch.argmax(y_hat, dim=1)
        accuracy = (preds == y).float().mean()

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_accuracy', accuracy, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)

        preds = torch.argmax(y_hat, dim=1)
        accuracy = (preds == y).float().mean()

        self.log('test_loss', loss)
        self.log('test_accuracy', accuracy)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Get Data

In [22]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os


path = os.path.join('Data', 'train.csv')
data = pd.read_csv(path)

target_name = 'NObeyesdad'
X, y = data.drop([target_name], axis=1), data[target_name]

print(f'Trian, Val and Test data size: {len(y)}')

Trian, Val and Test data size: 13284


# Hyperparameters

In [23]:
from pytorch_lightning import Trainer
from ray.tune.integration.pytorch_lightning import TuneReportCallback
from ray import tune
import warnings
warnings.filterwarnings("ignore")

def train_model(config, X, y):
    model = LitClassificationModel(input_dim=X.shape[1],
        n_hidden=config['n_hidden'],
        hidden_dim=config['hidden_dim'],
        output_dim=len(y.unique()),
        lr=config['lr']
    )

    datamodule = MyDataModule(X, y, batch_size=128)

    trainer = Trainer(
        max_epochs=20,
        log_every_n_steps=5,
        enable_checkpointing=False,
        callbacks=[TuneReportCallback({'loss': 'val_loss'}, on='validation_end')],
    )

    trainer.fit(model, datamodule)

# Ray Tune search
trainable_with_data = tune.with_parameters(train_model, X=X, y=y)

analysis = tune.run(
    trainable_with_data,
    config={
        'n_hidden': tune.choice([3,6,8,10]),
        'hidden_dim': tune.choice([32, 64, 128, 256, 512]),
        'lr': tune.loguniform(1e-4, 5e-1),
    },
    num_samples=20
)

2026-06-18 04:52:08,422	INFO tune.py:615 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(train_model pid=16801) /home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(train_model pid=16801) GPU available: False, used: False
(train_model pid=16801) TPU available: False, using: 0 TPU cores
(train_model pid=16801) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
(train_model pid=16801) 
(train_model pid=16801)   | Name          | Type             | Params | Mode  | FLOPs
(train_model pid=16801) -------------------------------------------------------------------
(train_model pid=16801) 0 | input_layer   | Linear           | 352    | train | 0    
(train_model pid=16801) 1 | hid

Epoch 0:  24%|██▍       | 16/67 [00:00<00:01, 43.67it/s, v_num=0]


(train_model pid=16806) 
(train_model pid=16806) 0 | input_layer   | Linear           | 2.8 K  | train | 0    
(train_model pid=16806) 2 | output_layer  | Linear           | 1.8 K  | train | 0    
(train_model pid=16807) 


Epoch 0:   0%|          | 0/67 [00:00<?, ?it/s]                            
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]


...

(train_model pid=16809) `Trainer.fit` stopped: `max_epochs=20` reached.


(train_model pid=16809) 
Epoch 19:  94%|█████████▍| 63/67 [00:03<00:00, 16.74it/s, v_num=0, val_loss=1.940, val_accuracy=0.138] [repeated 2x across cluster]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 10x across cluster]
Validation DataLoader 0: 100%|██████████| 17/17 [00:00<00:00, 93.00it/s] [repeated 8x across cluster]
(train_model pid=17767) 
Validation DataLoader 0:  12%|█▏        | 2/17 [00:00<00:00, 98.64it/s] 
(train_model pid=17610) 
(train_model pid=17610) 
Epoch 11:  94%|█████████▍| 63/67 [00:04<00:00, 13.31it/s, v_num=0, val_loss=1.940, val_accuracy=0.203]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 4x across cluster]
Epoch 12:  97%|█████████▋| 65/67 [00:04<00:00, 15.84it/s, v_num=0, val_loss=1.960, val_accuracy=0.138]
(train_model pid=17610) 
(train_model pid=17610) 
(train_model pid=17610) 
Epoch 13:  69%|██████▊   | 46/67 [00:02<00:01, 16.52it/s, v_num=0, val_loss=1.960, val_accuracy=0.121] [repeated 29x across cluster]
Validation: |          | 0/?

(train_model pid=17610) `Trainer.fit` stopped: `max_epochs=20` reached. [repeated 2x across cluster]
2026-06-18 04:55:27,066	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/home/franio/ray_results/train_model_2026-06-18_04-52-08' in 0.0078s.


(train_model pid=17610) 
Epoch 19: 100%|██████████| 67/67 [00:04<00:00, 15.65it/s, v_num=0, val_loss=1.970, val_accuracy=0.146]


2026-06-18 04:55:27,074	INFO tune.py:1033 -- Total run time: 198.65 seconds (198.56 seconds for the tuning loop).


In [24]:
print(analysis.get_best_config(metric='loss', mode='min'))

{'n_hidden': 3, 'hidden_dim': 128, 'lr': 0.0010853905341577717}


# Training

In [26]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping

config = analysis.get_best_config(metric='loss', mode='min')
model = LitClassificationModel(input_dim=X.shape[1], output_dim=len(y.unique()), **config)
datamodule = MyDataModule(X, y, batch_size=128)

early_stop_callback = EarlyStopping(
  monitor='val_loss',
  patience=5,
  mode='min',
  verbose=True
)

model_path = os.path.join('Models', 'NotCalibrated')
os.makedirs(model_path, exist_ok=True)


checkpoint_callback = ModelCheckpoint(
  dirpath=model_path,
  filename='model-{epoch:02d}-{val_loss:.2f}',
  monitor='val_loss',
  mode='min',
  save_top_k=3,
  save_last=True
)

trainer = Trainer(
  callbacks=[early_stop_callback,checkpoint_callback],
  max_epochs=100,
  accelerator='gpu',
  devices=1
 )

trainer.fit(model, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | input_layer   | Linear           | 1.4 K  | train | 0    
1 | hidden_layers | ModuleList       | 49.5 K | train | 0    
2 | output_layer  | Linear           | 903    | train | 0    
3 | activation    | ReLU             | 0      | train | 0    
4 | criterion     | CrossEntropyLoss | 0      | train | 0    
-------------------------------------------------------------------
51.8 K    Trainable params
0         Non-trainable params
51.8 K    Total params
0.207     Total estimated model params size (MB)
8        

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 0.574


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.436. Signaling Trainer to stop.


In [27]:
trainer.test(model, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.8415506482124329
        test_loss           0.44306740164756775
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.44306740164756775, 'test_accuracy': 0.8415506482124329}]